In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Aplicando paleta de cores
COLORS = {
    'primary': '#1e3a5f',      
    'secondary': '#20b2aa',     
    'accent': '#00ff7f',        
    'success': '#32cd32',       
    'warning': '#ffa500',    
    'danger': '#ff6b6b',     
    'info': '#4169e1',        
    'light': '#f8f9fa',       
    'dark': '#2c3e50',        
    'gradient_start': '#1e3a5f',
    'gradient_end': '#20b2aa'
}

plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': COLORS['dark'],
    'axes.linewidth': 1.2,
    'axes.labelcolor': COLORS['dark'],
    'text.color': COLORS['dark'],
    'xtick.color': COLORS['dark'],
    'ytick.color': COLORS['dark'],
    'grid.color': '#e0e0e0',
    'grid.alpha': 0.6,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10
})


In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas distintas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])


# Crescimento da demanda por centro de distribuição
Análise comparativa regional: províncias EUA vs Resto do Mundo

### Objetivo
Identificar qual grupo de centros de distribuição (províncias dos EUA ou países do resto do mundo) apresentou maior crescimento na demanda durante os próximos 3 meses previstos, utilizando regressão polimonial.

In [0]:
usa_countries = ['United States']
monthly_data['region_group'] = monthly_data['country_region_name'].apply(
    lambda x: 'províncias EUA' if x in usa_countries else 'Resto do mundo'
)

growth_results = {}
forecast_horizon = 3

for group in monthly_data['region_group'].unique():    
    group_monthly_totals = monthly_data[monthly_data['region_group'] == group].groupby(
        pd.Grouper(key='order_date', freq='M')
    )['order_quantity'].sum().sort_index()
    
    trend_data = group_monthly_totals.rolling(
        window=12,
        center=True,
        min_periods=6
    ).mean().fillna(method='bfill').fillna(method='ffill')

    X_trend = np.arange(len(trend_data)).reshape(-1, 1)
    
    poly_features = PolynomialFeatures(degree=2)
    X_poly = poly_features.fit_transform(X_trend)
    
    model = LinearRegression()
    model.fit(X_poly, trend_data.values)
    
    r2_fit = model.score(X_poly, trend_data.values)
    
    future_X = np.arange(len(trend_data), len(trend_data) + forecast_horizon).reshape(-1, 1)
    future_X_poly = poly_features.transform(future_X)
    trend_projection = np.maximum(model.predict(future_X_poly), 0)
    
    last_trend_value = trend_data.values[-1]
    last_projected_value = trend_projection[-1]
    
    projected_growth = ((last_projected_value - last_trend_value) / last_trend_value) * 100 if last_trend_value > 0 else 0
    
    growth_results[group] = {
        'projected_trend_growth': projected_growth,
        'r2_fit': r2_fit,
        'monthly_data': group_monthly_totals,
        'trend_data': trend_data,
        'trend_prediction': pd.Series(
            trend_projection,
            index=pd.date_range(group_monthly_totals.index[-1], periods=forecast_horizon + 1, freq='M')[1:]
        )
    }

    print(f"R²: {r2_fit:.3f}")
    print(f"Crescimento projetado da tendência para os próximos 3 meses: {projected_growth:.2f}%")


In [0]:
summary_df = pd.DataFrame(growth_results).T[['projected_trend_growth', 'r2_fit']]
summary_df = summary_df.sort_values(by='projected_trend_growth', ascending=False)
print(summary_df.to_string(formatters={
    'projected_trend_growth': '{:.2f}%'.format,
    'r2_fit': '{:.3f}'.format
}))

winner_region = summary_df.index[0]
winner_growth = summary_df.iloc[0]['projected_trend_growth']
print(f"{winner_region}' apresentou a maior tendência de crescimento projetada ({winner_growth:.2f}%).")


In [0]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(len(growth_results), 1, figsize=(16, 8 * len(growth_results)), squeeze=False)
axes = axes.flatten()

for i, group in enumerate(growth_results.keys()):
    ax = axes[i]
    data = growth_results[group]
    
    ax.plot(data['monthly_data'].index, data['monthly_data'].values, 'o-', color='gray', alpha=0.4, label='Vendas mensais reais')
    
    ax.plot(data['trend_data'].index, data['trend_data'].values, '-', color=COLORS['primary'], lw=2.5, label='Tendência central histórica')
    
    ax.plot(data['trend_prediction'].index, data['trend_prediction'].values, '--', color=COLORS['info'], lw=2.5, label=f"projeção da tendência ({data['projected_trend_growth']:.1f}%)")
    
    ax.axvline(data['monthly_data'].index[-1], color=COLORS['danger'], linestyle=':', label='Início da Previsão')

    ax.set_title(f"Tendência de crescimento para: {group} (R²: {data['r2_fit']:.3f})", fontweight='bold')
    ax.set_ylabel('Quantidade de vendas mensal')
    ax.legend(loc='upper left')

plt.suptitle('Comparação da tendência de crescimento', fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()